# Jane Street Market Forecasting - PyTorch LSTM Baseline (Offline, Comparable)

This notebook trains a compact LSTM baseline for `responder_6` while keeping the validation contract comparable to `train_lgbm_baseline.ipynb`:

- Same Polars-first data loading and memory-aware downcasting
- Same base and engineered feature set
- Same chronological fold strategy and date windows
- Same train-only median imputation per fold
- Same competition metric: **sample-weighted zero-mean R^2**

The model differs by batching one full trading day at a time: each sequence is a single symbol's intraday series (length `T_STEPS`, ~968), and the RNN predicts every `time_id` in the day (many-to-many), matching the winning solution's batching. Validation labels remain restricted to the same validation dates as the LightGBM baseline.

In [ ]:
# Uncomment and run once if needed.
# !pip install torch polars pyarrow pandas scikit-learn

import copy
import gc
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

In [ ]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "jane-street-market-forecasting":
    PROJECT_DIR = PROJECT_DIR / "jane-street-market-forecasting"

KAGGLE_DATA_DIR = Path("/kaggle/input/competitions/jane-street-real-time-market-data-forecasting")
IS_KAGGLE = KAGGLE_DATA_DIR.exists()

DATA_DIR = KAGGLE_DATA_DIR if IS_KAGGLE else (PROJECT_DIR / "data")
TRAIN_DIR = DATA_DIR / "train.parquet"
TRAIN_GLOB = str(TRAIN_DIR / "partition_id=*" / "*.parquet")

# Cache for the feature-engineered frame so it can be streamed per-fold
# instead of being held in RAM for the whole training loop.
FE_CACHE_DIR = Path("/kaggle/working") if IS_KAGGLE else (PROJECT_DIR / "data")
FE_PARQUET_PATH = FE_CACHE_DIR / "fe_cache.parquet"

TARGET = "responder_6"
WEIGHT_COL = "weight"
FEATURE_COLS = [f"feature_{i:02d}" for i in range(79)]
RESPONDER_COLS = [f"responder_{i}" for i in range(9)]
BASE_COLS = ["date_id", "time_id", "symbol_id", WEIGHT_COL] + FEATURE_COLS + RESPONDER_COLS

# Single fold matching the winner's "Fold 1 with 200 days gap":
#   train  : TRAIN_START_DATE .. TRAIN_END_DATE   (~700..1298)
#   gap    : TRAIN_END_DATE+1 .. VAL_START_DATE-1 (1299..1498, NOT loaded)
#   val    : VAL_START_DATE .. VAL_END_DATE       (1499..1698)
# The gap dates are never loaded, which keeps memory close to a normal 2-window run.
TRAIN_START_DATE = 800
TRAIN_END_DATE = 1298
VAL_START_DATE = 1499
VAL_END_DATE = 1698

# Match the LGBM baseline feature switches.
PHASE1_ENABLE = True
# Hardcoded correlated base features (from the winning solution's COLS_FEATURES_CORR).
# Using a fixed list avoids a memory-spiking correlation pass over a full
# train-subset copy of the data.
PHASE1_BASE_FEATURES = [
    # "feature_06", "feature_04", "feature_07", "feature_36",
    "feature_60", "feature_45", "feature_56", "feature_05",
    # "feature_51", "feature_19", "feature_66", "feature_59",
    # "feature_54", "feature_70", "feature_71", "feature_72",
]

PHASE2_ENABLE = False
PHASE2_TOPK_BASE_FEATURES = 8
PHASE2_LAGS = [1, 2]
PHASE3_ENABLE = False
PHASE3_USE_RESPONDER_LAGS = True

# Winner-style batching: each sequence is one full trading day for a symbol, and
# the model predicts EVERY time_id in that day (many-to-many). The day length
# (T_STEPS) is derived from the data in the feature cell (stable at 968 from
# date_id >= 700). BATCH_SIZE is measured in DAYS: one day batches all of that
# day's symbols together (~tens of sequences x ~968 steps). The winner used 1.
BATCH_SIZE = 1
HIDDEN_SIZE = 64
NUM_LAYERS = 1
DROPOUT = 0.1
USE_GRU = True  # set True to train a GRU instead of an LSTM
SYMBOL_EMBED_DIM = 8
STANDARDIZE_INPUTS = True  # z-score SCALE_FEATURES only (train-fold stats)
LR = 1e-4
WEIGHT_DECAY = 1e-4
# Training objective. "weighted_zero_mean_r2" optimizes the competition metric
# directly; "weighted_mse" is the previous behavior.
LOSS = "weighted_zero_mean_r2"

# Auxiliary targets (multi-task), following the winning solution: the model
# predicts these responders alongside responder_6 to regularize the shared
# representation, and the per-responder weighted zero-mean R^2 losses are summed
# (loss_main + sum(loss_aux)). Only responder_6 is used for the metric and the
# final prediction; the aux heads exist purely to shape the trunk. responder_7
# and responder_8 are ~120-day and ~4-day rolling averages of the same latent
# variable as responder_6, so they carry related signal.
AUX_TARGETS_ENABLE = True
AUX_RESPONDERS = ["responder_7", "responder_8"]
AUX_LOSS_WEIGHT = 1.0

MAX_EPOCHS = 100
PATIENCE = 6
LR_PLATEAU_FACTOR = 0.5
LR_PLATEAU_PATIENCE = 3
LR_MIN = 1e-6
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

# Set True for a very quick shape/training smoke test (uses only a few recent
# days per split). Keep False for comparable runs.
FAST_DEV_RUN = False

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)

print(f"Running on Kaggle: {IS_KAGGLE}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"DATA_DIR exists: {DATA_DIR.exists()}")
print(f"TRAIN_DIR exists: {TRAIN_DIR.exists()}")
print(f"TRAIN window: [{TRAIN_START_DATE}, {TRAIN_END_DATE}]")
print(f"VAL window: [{VAL_START_DATE}, {VAL_END_DATE}]")
print(f"BATCH_SIZE (days): {BATCH_SIZE}")
print(f"SYMBOL_EMBED_DIM: {SYMBOL_EMBED_DIM}")
print(f"AUX_TARGETS_ENABLE: {AUX_TARGETS_ENABLE} | AUX_RESPONDERS: {AUX_RESPONDERS if AUX_TARGETS_ENABLE else []}")
print(f"DEVICE: {DEVICE}")
print(f"FAST_DEV_RUN: {FAST_DEV_RUN}")

In [ ]:
def build_scan() -> pl.LazyFrame:
    scan = pl.scan_parquet(TRAIN_GLOB).select(BASE_COLS)

    # Load only the train and val date ranges; the 200-day gap in between is
    # never read, so peak memory stays close to a normal two-window run.
    scan = scan.filter(
        pl.col("date_id").is_between(TRAIN_START_DATE, TRAIN_END_DATE)
        | pl.col("date_id").is_between(VAL_START_DATE, VAL_END_DATE)
    )

    # Downcast to reduce memory pressure while preserving practical precision.
    cast_exprs = [
        pl.col("date_id").cast(pl.Int16),
        pl.col("time_id").cast(pl.Int16),
        pl.col("symbol_id").cast(pl.Int16),
        pl.col(WEIGHT_COL).cast(pl.Float32),
    ] + [pl.col(c).cast(pl.Float32) for c in FEATURE_COLS + RESPONDER_COLS]

    scan = scan.with_columns(cast_exprs)
    return scan


scan = build_scan()
df_pl = scan.collect(streaming=True).sort(["date_id", "time_id", "symbol_id"])

print(df_pl.shape)
print(df_pl.select(["date_id", "time_id", "symbol_id", WEIGHT_COL, TARGET]).head())
print(f"Estimated in-memory size (MB): {df_pl.estimated_size('mb'):.2f}")

In [ ]:
def add_basic_features_polars(frame: pl.DataFrame) -> pl.DataFrame:
    max_time = max(int(frame["time_id"].max()), 1)
    feature_exprs = [pl.col(c) for c in FEATURE_COLS]

    out = frame.with_columns(
        [
            pl.sum_horizontal([pl.col(c).is_null().cast(pl.Int16) for c in FEATURE_COLS])
            .cast(pl.Int16)
            .alias("feature_nan_count"),
            pl.mean_horizontal(feature_exprs).cast(pl.Float32).alias("feature_row_mean"),
            pl.mean_horizontal([pl.col(c).abs() for c in FEATURE_COLS])
            .cast(pl.Float32)
            .alias("feature_row_abs_mean"),
            ((2.0 * np.pi * pl.col("time_id").cast(pl.Float32)) / float(max_time))
            .sin()
            .cast(pl.Float32)
            .alias("time_sin"),
            ((2.0 * np.pi * pl.col("time_id").cast(pl.Float32)) / float(max_time))
            .cos()
            .cast(pl.Float32)
            .alias("time_cos"),
        ]
    )
    return out


def select_top_base_features_by_abs_corr(
    frame: pl.DataFrame,
    candidate_cols: list[str],
    target_col: str,
    top_k: int = 10,
) -> list[str]:
    corr_exprs = [pl.corr(pl.col(c), pl.col(target_col)).abs().alias(c) for c in candidate_cols]
    corr_row = frame.select(corr_exprs).row(0, named=True)

    sorted_feats = sorted(
        candidate_cols,
        key=lambda c: (corr_row[c] if corr_row[c] is not None else -1.0),
        reverse=True,
    )
    return sorted_feats[:top_k]


def add_cross_sectional_features_polars(
    frame: pl.DataFrame,
    selected_base_features: list[str],
) -> pl.DataFrame:
    group_cols = ["date_id", "time_id"]
    cs_exprs = []

    for c in selected_base_features:
        cs_exprs.append(
            (pl.col(c) - pl.col(c).mean().over(group_cols))
            .cast(pl.Float32)
            .alias(f"{c}_cs_demean")
        )
        cs_exprs.append(
            (
                pl.col(c).rank("average").over(group_cols).cast(pl.Float32)
                / pl.len().over(group_cols).cast(pl.Float32)
            )
            .cast(pl.Float32)
            .alias(f"{c}_cs_rank_pct")
        )

    return frame.with_columns(cs_exprs)


def add_prev_day_responder_lags_polars(
    frame: pl.DataFrame,
    responder_cols: list[str],
) -> pl.DataFrame:
    # Emulate lags.parquet availability: previous-date responder values by symbol.
    day_last_exprs = [
        pl.col(c).last().cast(pl.Float32).alias(f"{c}_day_last")
        for c in responder_cols
    ]

    daily = frame.group_by(["symbol_id", "date_id"]).agg(day_last_exprs).sort(["symbol_id", "date_id"])

    lag_cols = [f"lag1d_{c}" for c in responder_cols]
    lag_exprs = [
        pl.col(f"{c}_day_last")
        .shift(1)
        .over("symbol_id")
        .cast(pl.Float32)
        .alias(f"lag1d_{c}")
        for c in responder_cols
    ]

    lag_df = daily.with_columns(lag_exprs).select(["symbol_id", "date_id"] + lag_cols)

    out = frame.join(lag_df, on=["symbol_id", "date_id"], how="left")

    out = out.with_columns(
        [
            pl.mean_horizontal([pl.col(c) for c in lag_cols])
            .cast(pl.Float32)
            .alias("lag1d_resp_mean"),
            pl.mean_horizontal([pl.col(c).abs() for c in lag_cols])
            .cast(pl.Float32)
            .alias("lag1d_resp_abs_mean"),
            (
                pl.col("lag1d_responder_6")
                - pl.mean_horizontal([pl.col(c) for c in lag_cols])
            )
            .cast(pl.Float32)
            .alias("lag1d_target_vs_resp_mean"),
        ]
    )

    return out


def weighted_zero_mean_r2(y_true: np.ndarray, y_pred: np.ndarray, w: np.ndarray) -> float:
    num = np.sum(w * (y_true - y_pred) ** 2)
    den = np.sum(w * (y_true**2))
    if den == 0:
        return np.nan
    return 1.0 - num / den


def make_time_series_folds(
    unique_dates: np.ndarray,
    n_folds: int = 3,
    val_days: int = 180,
    gap: int = 1,
    train_lookback_days: int | None = 540,
):
    """
    Build chronological folds with fixed-size validation blocks near the dataset tail.
    Each fold validation span is `val_days` (about 6 months).
    """
    n_dates = len(unique_dates)
    required = n_folds * val_days + gap + 30
    if n_dates < required:
        raise ValueError(
            f"Not enough dates ({n_dates}) for {n_folds} folds with val_days={val_days}."
        )

    folds = []
    for k in range(n_folds):
        # Older fold first, newest fold last.
        val_end = n_dates - (n_folds - 1 - k) * val_days
        val_start = val_end - val_days
        train_end = val_start - gap

        if train_lookback_days is None:
            train_start = 0
        else:
            train_start = max(0, train_end - train_lookback_days)

        train_dates = unique_dates[train_start:train_end]
        val_dates = unique_dates[val_start:val_end]

        if len(train_dates) == 0 or len(val_dates) == 0:
            raise ValueError("Empty train/val split created. Increase MAX_DATES.")

        folds.append((train_dates, val_dates))

    return folds

In [ ]:
# Build baseline engineered dataset first.
if "df_pl" not in globals():
    scan = build_scan()
    df_pl = scan.collect(streaming=True).sort(["date_id", "time_id", "symbol_id"])

df_base_pl = add_basic_features_polars(df_pl)
del df_pl
gc.collect()

unique_dates = np.sort(df_base_pl["date_id"].unique().to_numpy())
train_dates = unique_dates[
    (unique_dates >= TRAIN_START_DATE) & (unique_dates <= TRAIN_END_DATE)
]
val_dates = unique_dates[
    (unique_dates >= VAL_START_DATE) & (unique_dates <= VAL_END_DATE)
]
if len(train_dates) == 0 or len(val_dates) == 0:
    raise ValueError("Empty train or val window; check the date range constants.")

# Single fold matching the winner's "Fold 1 with 200 days gap".
folds = [(train_dates, val_dates)]

gap_days = int(val_dates.min()) - int(train_dates.max()) - 1
print(
    f"Fold 1: train [{int(train_dates.min())}, {int(train_dates.max())}] "
    f"({len(train_dates)} dates) | "
    f"val [{int(val_dates.min())}, {int(val_dates.max())}] ({len(val_dates)} dates) | "
    f"gap={gap_days} dates"
)

if PHASE1_ENABLE:
    # Hardcoded correlated features: no correlation pass and no full
    # train-subset copy, which is what spiked RAM before.
    phase1_top_features = [c for c in PHASE1_BASE_FEATURES if c in df_base_pl.columns]
    print(f"Phase 1 base features ({len(phase1_top_features)}): {phase1_top_features}")

    df_fe_pl = add_cross_sectional_features_polars(df_base_pl, phase1_top_features)
else:
    phase1_top_features = []
    df_fe_pl = df_base_pl

if PHASE3_ENABLE:
    if PHASE3_USE_RESPONDER_LAGS:
        df_fe_pl = add_prev_day_responder_lags_polars(df_fe_pl, RESPONDER_COLS)

# Drop intermediate baseline frame once final feature frame is built.
if df_fe_pl is not df_base_pl:
    del df_base_pl
gc.collect()

EXCLUDE_FROM_MODEL = {TARGET, WEIGHT_COL, "date_id", "symbol_id"} # time_id
MODEL_FEATURES = [
    c
    for c in df_fe_pl.columns
    if c not in EXCLUDE_FROM_MODEL and not c.startswith("responder_")
]

# Z-score only continuous market/lag features; keep cyclical/count columns raw.
NO_SCALE_FEATURES = {"time_id", "time_sin", "time_cos", "feature_nan_count"}
SCALE_FEATURES = [c for c in MODEL_FEATURES if c not in NO_SCALE_FEATURES]
SCALE_COL_INDICES = np.array([MODEL_FEATURES.index(c) for c in SCALE_FEATURES], dtype=np.int64)
PASSTHROUGH_FEATURES = [c for c in MODEL_FEATURES if c not in SCALE_FEATURES]

print(f"Model features ({len(MODEL_FEATURES)}):")
for c in MODEL_FEATURES:
    tag = " [z-score]" if c in SCALE_FEATURES else " [raw]"
    print(f"  {c}{tag}")
print(f"\nZ-score columns ({len(SCALE_FEATURES)}): {SCALE_FEATURES}")
print(f"Raw passthrough columns ({len(PASSTHROUGH_FEATURES)}): {PASSTHROUGH_FEATURES}")
print(f"Feature-engineered size (MB): {df_fe_pl.estimated_size('mb'):.2f}")

# Day length for the many-to-many sequences: number of intraday time steps.
# Stable at 968 from date_id >= 700 (the winner hardcodes T = 968).
T_STEPS = int(df_fe_pl["time_id"].max()) + 1
N_SYMBOLS = int(df_fe_pl["symbol_id"].max()) + 1
print(f"T_STEPS (time_ids per day): {T_STEPS} | N_SYMBOLS: {N_SYMBOLS}")

# Auxiliary responder targets for multi-task training (never used as inputs).
if AUX_TARGETS_ENABLE:
    AUX_TARGET_COLS = [c for c in AUX_RESPONDERS if c in df_fe_pl.columns]
    missing_aux = [c for c in AUX_RESPONDERS if c not in df_fe_pl.columns]
    if missing_aux:
        raise ValueError(f"Requested aux responders not in frame: {missing_aux}")
else:
    AUX_TARGET_COLS = []
print(f"Auxiliary target columns ({len(AUX_TARGET_COLS)}): {AUX_TARGET_COLS}")

# Columns the per-fold loop actually needs (drop unused responders, etc.).
COLS_FOR_FOLD = list(
    dict.fromkeys(
        MODEL_FEATURES
        + [TARGET, WEIGHT_COL, "symbol_id", "date_id", "time_id"]
        + AUX_TARGET_COLS
    )
)

# Persist the engineered frame to disk and free it from RAM. The training loop
# streams each fold back via a predicate-pushdown scan instead of keeping the
# full feature-engineered frame resident.
FE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
df_fe_pl.select(COLS_FOR_FOLD).write_parquet(FE_PARQUET_PATH)
print(f"Wrote feature cache: {FE_PARQUET_PATH}")

del df_fe_pl
gc.collect()

In [ ]:
def compute_train_medians(train_pl: pl.DataFrame, feature_cols: list[str]) -> dict[str, float]:
    medians = train_pl.select([pl.col(c).median().alias(c) for c in feature_cols]).row(0, named=True)
    clean_medians = {}
    for c, value in medians.items():
        if value is None or not np.isfinite(value):
            clean_medians[c] = 0.0
        else:
            clean_medians[c] = float(value)
    return clean_medians


def fill_features_with_medians(frame: pl.DataFrame, feature_cols: list[str], medians: dict[str, float]) -> pl.DataFrame:
    fill_exprs = [
        pl.col(c)
        .fill_null(medians[c])
        .fill_nan(medians[c])
        .cast(pl.Float32)
        .alias(c)
        for c in feature_cols
    ]
    return frame.with_columns(fill_exprs)


class PerDayDataset(Dataset):
    """One item = one date, holding every symbol's full-day sequence.

    Mirrors the winning solution's batching: data is grouped by ``date_id`` and a
    single item returns a dense ``(n_symbols_in_day, T_steps, n_features)`` tensor
    so the RNN runs over the intraday time axis and predicts every ``time_id``
    (many-to-many). Missing ``(symbol, time)`` slots are zero-filled with weight 0,
    so the weighted loss/metric ignore them.

    The dense day tensor is built lazily in ``__getitem__`` (like the winner's
    ``on_batch`` path) to avoid materializing the whole padded cube at once.
    """

    def __init__(
        self,
        X: np.ndarray,
        y: np.ndarray,
        w: np.ndarray,
        resp: np.ndarray,
        symbol_ids: np.ndarray,
        date_ids: np.ndarray,
        time_ids: np.ndarray,
        t_steps: int,
    ) -> None:
        # Inputs are assumed already sorted by date_id (see prepare_fold_arrays),
        # so we store references without re-sorting/copying the large arrays.
        self.X = X
        self.y = y
        self.w = w
        # Auxiliary targets, shape (N, n_aux); width 0 when aux targets are off.
        self.resp = resp
        self.symbol_ids = symbol_ids.astype(np.int64, copy=False)
        self.time_ids = time_ids.astype(np.int64, copy=False)
        self.t_steps = int(t_steps)
        self.n_features = X.shape[1]
        self.n_aux = resp.shape[1]

        self.unique_dates, starts = np.unique(date_ids, return_index=True)
        self.starts = starts.astype(np.int64)
        self.ends = np.append(starts[1:], len(date_ids)).astype(np.int64)

    def __len__(self) -> int:
        return len(self.unique_dates)

    def __getitem__(self, idx: int):
        s, e = int(self.starts[idx]), int(self.ends[idx])
        sids = self.symbol_ids[s:e]
        tids = self.time_ids[s:e]

        uniq_sym, sym_local = np.unique(sids, return_inverse=True)
        n_sym = len(uniq_sym)
        T = self.t_steps

        X_day = np.zeros((n_sym, T, self.n_features), dtype=np.float32)
        y_day = np.zeros((n_sym, T), dtype=np.float32)
        w_day = np.zeros((n_sym, T), dtype=np.float32)
        resp_day = np.zeros((n_sym, T, self.n_aux), dtype=np.float32)

        valid = (tids >= 0) & (tids < T)
        rows = np.arange(s, e)[valid]
        sl = sym_local[valid]
        tl = tids[valid]
        X_day[sl, tl] = self.X[rows]
        y_day[sl, tl] = self.y[rows]
        w_day[sl, tl] = self.w[rows]
        if self.n_aux > 0:
            resp_day[sl, tl] = self.resp[rows]

        return (
            torch.from_numpy(X_day),
            torch.from_numpy(uniq_sym.astype(np.int64)),
            torch.from_numpy(y_day),
            torch.from_numpy(w_day),
            torch.from_numpy(resp_day),
        )


def per_day_collate(batch: list):
    """Concatenate per-day tensors along the symbol axis (batch in days)."""
    X = torch.cat([b[0] for b in batch], dim=0)
    sid = torch.cat([b[1] for b in batch], dim=0)
    y = torch.cat([b[2] for b in batch], dim=0)
    w = torch.cat([b[3] for b in batch], dim=0)
    resp = torch.cat([b[4] for b in batch], dim=0)
    return X, sid, y, w, resp


def prepare_fold_arrays(
    frame: pl.DataFrame,
    feature_cols: list[str],
    aux_cols: list[str] | None = None,
):
    # Sort by date so the per-day dataset can group rows without re-copying.
    sort_cols = ["date_id", "time_id", "symbol_id"]
    frame = frame.sort(sort_cols)

    X = np.ascontiguousarray(frame.select(feature_cols).to_numpy(), dtype=np.float32)
    y = np.ascontiguousarray(frame[TARGET].to_numpy(), dtype=np.float32)
    w = np.ascontiguousarray(frame[WEIGHT_COL].to_numpy(), dtype=np.float32)
    symbol_ids = np.ascontiguousarray(frame["symbol_id"].to_numpy(), dtype=np.int64)
    date_ids = np.ascontiguousarray(frame["date_id"].to_numpy(), dtype=np.int64)
    time_ids = np.ascontiguousarray(frame["time_id"].to_numpy(), dtype=np.int64)

    # Auxiliary targets: NaN-filled with 0 (matching the winner's responder
    # handling); rows with no signal get weight 0 via the day-tensor masking.
    if aux_cols:
        resp = np.ascontiguousarray(frame.select(aux_cols).to_numpy(), dtype=np.float32)
        resp = np.nan_to_num(resp, nan=0.0)
    else:
        resp = np.zeros((len(y), 0), dtype=np.float32)

    return X, y, w, resp, symbol_ids, date_ids, time_ids


def fit_standardizer(X: np.ndarray, col_indices: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Per-feature mean/std on selected columns only; other columns use mean=0, std=1."""
    mean = np.zeros((1, X.shape[1]), dtype=np.float32)
    std = np.ones((1, X.shape[1]), dtype=np.float32)
    if len(col_indices) == 0:
        return mean, std

    subset = X[:, col_indices]
    col_mean = subset.mean(axis=0).astype(np.float32)
    col_std = subset.std(axis=0)
    col_std = np.where(col_std < 1e-8, 1.0, col_std).astype(np.float32)

    mean[0, col_indices] = col_mean
    std[0, col_indices] = col_std
    return mean, std


def apply_standardizer(X: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return np.ascontiguousarray((X - mean) / std, dtype=np.float32)


def maybe_limit_dates(dates: np.ndarray, max_dates: int | None) -> np.ndarray:
    """Keep only the most recent `max_dates` dates (for FAST_DEV smoke tests)."""
    if max_dates is None or len(dates) <= max_dates:
        return dates
    return dates[-max_dates:]


def make_loader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool,
    collate_fn=per_day_collate,
) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=(DEVICE == "cuda"),
        drop_last=False,
        collate_fn=collate_fn,
    )

In [ ]:
class SequenceRegressor(nn.Module):
    def __init__(
        self,
        n_features: int,
        n_symbols: int,
        symbol_embed_dim: int = 8,
        hidden_size: int = 128,
        num_layers: int = 1,
        dropout: float = 0.10,
        use_gru: bool = False,
        n_aux: int = 0,
    ) -> None:
        super().__init__()
        self.symbol_embed = nn.Embedding(n_symbols, symbol_embed_dim)

        rnn_dropout = dropout if num_layers > 1 else 0.0
        rnn_cls = nn.GRU if use_gru else nn.LSTM
        self.rnn = rnn_cls(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=rnn_dropout,
            batch_first=True,
        )
        head_in = hidden_size + symbol_embed_dim

        def make_head() -> nn.Module:
            return nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(head_in, hidden_size // 2),
                nn.SiLU(),
                nn.Linear(hidden_size // 2, 1),
            )

        # One head per responder over a shared RNN trunk: the main responder_6
        # head drives the metric, the aux heads (responder_7/8) only regularize
        # the trunk. (The winning solution uses a separate RNN per responder; we
        # share the trunk here since this baseline is intentionally compact.)
        self.n_aux = n_aux
        self.head = make_head()
        self.aux_heads = nn.ModuleList([make_head() for _ in range(n_aux)])

    def forward(
        self, x: torch.Tensor, symbol_id: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        # x: (B, T, F) where B = symbols-in-batch, T = intraday time steps.
        # Many-to-many: emit one prediction per time step -> (B, T).
        out, _ = self.rnn(x)                                  # (B, T, H)
        symbol_emb = self.symbol_embed(symbol_id)             # (B, E)
        symbol_emb = symbol_emb.unsqueeze(1).expand(-1, out.size(1), -1)  # (B, T, E)
        feat = torch.cat([out, symbol_emb], dim=-1)           # (B, T, H+E)
        main = self.head(feat).squeeze(-1)                    # (B, T)
        if self.n_aux == 0:
            return main, None
        aux = torch.stack(
            [h(feat).squeeze(-1) for h in self.aux_heads], dim=-1
        )                                                     # (B, T, n_aux)
        return main, aux


def weighted_mse_loss(pred: torch.Tensor, target: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:
    return (weight * (target - pred).pow(2)).sum() / weight.sum().clamp_min(1e-12)


def weighted_zero_mean_r2_loss(pred: torch.Tensor, target: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:
    # Competition metric is R2 = 1 - sum(w*(y-pred)^2) / sum(w*y^2).
    # Maximizing R2 == minimizing the weighted residual normalized by the
    # (batch-constant) weighted target energy. Unlike plain weighted MSE, this
    # rescales each batch by sum(w*y^2), aligning the gradient with the metric.
    num = (weight * (target - pred).pow(2)).sum()
    den = (weight * target.pow(2)).sum().clamp_min(1e-12)
    return num / den


LOSS_FNS = {
    "weighted_mse": weighted_mse_loss,
    "weighted_zero_mean_r2": weighted_zero_mean_r2_loss,
}


def get_loss_fn(name: str):
    if name not in LOSS_FNS:
        raise ValueError(f"Unknown LOSS={name!r}; choose from {sorted(LOSS_FNS)}.")
    return LOSS_FNS[name]


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer | None = None,
) -> dict[str, float]:
    is_train = optimizer is not None
    model.train(is_train)

    loss_fn = get_loss_fn(LOSS)

    total_loss_num = 0.0
    total_weight = 0.0
    total_r2_num = 0.0
    total_r2_den = 0.0

    for X_batch, sid_batch, y_batch, w_batch, resp_batch in loader:
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        sid_batch = sid_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)
        w_batch = w_batch.to(DEVICE, non_blocking=True)
        resp_batch = resp_batch.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            pred, aux_pred = model(X_batch, sid_batch)
            loss = loss_fn(pred, y_batch, w_batch)

            # Sum the per-responder losses (winner-style multi-task objective).
            # Only the main responder_6 head is used for the reported metric.
            if aux_pred is not None:
                for j in range(aux_pred.shape[-1]):
                    loss = loss + AUX_LOSS_WEIGHT * loss_fn(
                        aux_pred[..., j], resp_batch[..., j], w_batch
                    )

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        with torch.no_grad():
            err2 = (y_batch - pred).pow(2)
            total_loss_num += float((w_batch * err2).sum().detach().cpu())
            total_weight += float(w_batch.sum().detach().cpu())
            total_r2_num += float((w_batch * err2).sum().detach().cpu())
            total_r2_den += float((w_batch * y_batch.pow(2)).sum().detach().cpu())

    weighted_loss = np.nan if total_weight == 0 else total_loss_num / total_weight
    weighted_r2 = np.nan if total_r2_den == 0 else 1.0 - (total_r2_num / total_r2_den)
    return {
        "weighted_mse": weighted_loss,
        "weighted_zero_mean_r2": weighted_r2,
        "r2_num": total_r2_num,
        "r2_den": total_r2_den,
    }


def train_one_fold(
    train_dataset: Dataset,
    val_dataset: Dataset,
    n_features: int,
    n_symbols: int,
) -> tuple[nn.Module, dict[str, float], dict[str, float], int]:
    model = SequenceRegressor(
        n_features=n_features,
        n_symbols=n_symbols,
        symbol_embed_dim=SYMBOL_EMBED_DIM,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        use_gru=USE_GRU,
        n_aux=len(AUX_TARGET_COLS),
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=LR_PLATEAU_FACTOR,
        patience=LR_PLATEAU_PATIENCE,
        min_lr=LR_MIN,
    )

    train_loader = make_loader(train_dataset, BATCH_SIZE, shuffle=True)
    # Validate one day at a time, as in the winning solution.
    val_loader = make_loader(val_dataset, 1, shuffle=False)

    best_state = copy.deepcopy(model.state_dict())
    best_val_r2 = -np.inf
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_metrics = run_epoch(model, train_loader, optimizer=optimizer)
        val_metrics = run_epoch(model, val_loader, optimizer=None)
        scheduler.step(val_metrics["weighted_zero_mean_r2"])
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch:02d} | "
            f"lr={current_lr:.2e} | "
            f"train_loss={train_metrics['weighted_mse']:.6f} | "
            f"train_r2={train_metrics['weighted_zero_mean_r2']:.6f} | "
            f"val_loss={val_metrics['weighted_mse']:.6f} | "
            f"val_r2={val_metrics['weighted_zero_mean_r2']:.6f}"
        )

        if val_metrics["weighted_zero_mean_r2"] > best_val_r2:
            best_val_r2 = val_metrics["weighted_zero_mean_r2"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print(f"Early stopping at epoch {epoch}; best_epoch={best_epoch}.")
                break

        if FAST_DEV_RUN:
            print("FAST_DEV_RUN enabled; stopping after one epoch.")
            break

    model.load_state_dict(best_state)
    final_train_metrics = run_epoch(model, train_loader, optimizer=None)
    final_val_metrics = run_epoch(model, val_loader, optimizer=None)
    return model, final_train_metrics, final_val_metrics, best_epoch

In [ ]:
fold_metrics = []
global_val_num = 0.0
global_val_den = 0.0

for fold_idx, (train_dates, val_dates) in enumerate(folds, start=1):
    if FAST_DEV_RUN:
        # Smoke test: only the most recent few days per split.
        train_dates = maybe_limit_dates(train_dates, 20)
        val_dates = maybe_limit_dates(val_dates, 10)

    # Stream only this fold's rows/columns from disk (predicate + projection
    # pushdown) so the full feature-engineered frame never sits in RAM.
    fold_scan = pl.scan_parquet(FE_PARQUET_PATH).select(COLS_FOR_FOLD)
    train_pl = fold_scan.filter(pl.col("date_id").is_in(train_dates)).collect()
    val_pl = fold_scan.filter(pl.col("date_id").is_in(val_dates)).collect()

    medians = compute_train_medians(train_pl, MODEL_FEATURES)
    train_pl = fill_features_with_medians(train_pl, MODEL_FEATURES, medians)
    val_pl = fill_features_with_medians(val_pl, MODEL_FEATURES, medians)

    print(
        f"Fold {fold_idx} rows | train={train_pl.height:,}, val={val_pl.height:,} | "
        f"train_mb~{train_pl.estimated_size('mb'):.1f}, val_mb~{val_pl.estimated_size('mb'):.1f}"
    )

    X_train, y_train, w_train, resp_train, sid_train, date_train, time_train = prepare_fold_arrays(
        train_pl, MODEL_FEATURES, AUX_TARGET_COLS
    )
    X_val, y_val, w_val, resp_val, sid_val, date_val, time_val = prepare_fold_arrays(
        val_pl, MODEL_FEATURES, AUX_TARGET_COLS
    )

    del train_pl, val_pl
    gc.collect()

    if STANDARDIZE_INPUTS:
        feat_mean, feat_std = fit_standardizer(X_train, SCALE_COL_INDICES)
        X_train = apply_standardizer(X_train, feat_mean, feat_std)
        X_val = apply_standardizer(X_val, feat_mean, feat_std)

    print(f"Fold {fold_idx} model inputs ({len(MODEL_FEATURES)}): {MODEL_FEATURES}")
    if STANDARDIZE_INPUTS:
        print(f"Fold {fold_idx} z-scored columns ({len(SCALE_FEATURES)}): {SCALE_FEATURES}")

    train_dataset = PerDayDataset(
        X_train, y_train, w_train, resp_train, sid_train, date_train, time_train, T_STEPS
    )
    val_dataset = PerDayDataset(
        X_val, y_val, w_val, resp_val, sid_val, date_val, time_val, T_STEPS
    )

    print(
        f"Fold {fold_idx} days | train={len(train_dataset):,}, val={len(val_dataset):,} | "
        f"n_features={len(MODEL_FEATURES)} | T_STEPS={T_STEPS}"
    )

    if len(train_dataset) == 0 or len(val_dataset) == 0:
        raise ValueError(
            f"Fold {fold_idx} has no train or validation days. Check the date windows."
        )

    n_symbols_fold = N_SYMBOLS

    model, train_metrics, val_metrics, best_epoch = train_one_fold(
        train_dataset,
        val_dataset,
        n_features=len(MODEL_FEATURES),
        n_symbols=n_symbols_fold,
    )

    global_val_num += val_metrics["r2_num"]
    global_val_den += val_metrics["r2_den"]

    fold_metrics.append(
        {
            "fold": fold_idx,
            "best_epoch": best_epoch,
            "train_weighted_zero_mean_r2": train_metrics["weighted_zero_mean_r2"],
            "val_weighted_zero_mean_r2": val_metrics["weighted_zero_mean_r2"],
            "train_weighted_mse": train_metrics["weighted_mse"],
            "val_weighted_mse": val_metrics["weighted_mse"],
            "train_sequences": len(train_dataset),
            "val_sequences": len(val_dataset),
        }
    )

    print(
        f"Fold {fold_idx} | best_epoch={best_epoch} | "
        f"train_r2={train_metrics['weighted_zero_mean_r2']:.6f} | "
        f"val_r2={val_metrics['weighted_zero_mean_r2']:.6f}"
    )

    del (
        X_train,
        y_train,
        w_train,
        resp_train,
        sid_train,
        date_train,
        time_train,
        X_val,
        y_val,
        w_val,
        resp_val,
        sid_val,
        date_val,
        time_val,
        train_dataset,
        val_dataset,
        model,
    )
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

metrics_df = pd.DataFrame(fold_metrics)
metrics_df

In [ ]:
overall_oof_r2 = np.nan if global_val_den == 0 else 1.0 - (global_val_num / global_val_den)

print("\nTrain/Eval metrics summary")
print(metrics_df.to_string(index=False))
print(f"\nMean fold train weighted zero-mean R^2: {metrics_df['train_weighted_zero_mean_r2'].mean():.6f}")
print(f"Mean fold val weighted zero-mean R^2: {metrics_df['val_weighted_zero_mean_r2'].mean():.6f}")
print(f"Overall OOF weighted zero-mean R^2: {overall_oof_r2:.6f}")

## Gradual Tuning Notes

Start with the defaults above and compare `val_weighted_zero_mean_r2` plus `overall_oof_r2` to the LightGBM baseline.

Sequences are now one full trading day per symbol (length `T_STEPS`, ~968), and the
model predicts every `time_id` in the day (many-to-many), matching the winning
solution. `BATCH_SIZE` is in **days**.

Suggested tuning order:

1. Batch size (days): start at `1` (winner's setting), raise to `4`/`8`/`16` while it fits in memory.
2. Hidden size: `64`, `128`, `256`.
3. Depth: try `NUM_LAYERS = 2` and keep dropout enabled inside the RNN.
4. Learning rate: try `5e-4` and `2e-3` after picking stable capacity.

Auxiliary targets (multi-task): `AUX_TARGETS_ENABLE` adds one prediction head per
responder in `AUX_RESPONDERS` (default `responder_7`, `responder_8`) over the shared
RNN trunk. The training objective is the sum of per-responder weighted zero-mean R^2
losses (`loss_main + AUX_LOSS_WEIGHT * sum(loss_aux)`), following the winning
solution; only `responder_6` is used for the reported metric. Set
`AUX_TARGETS_ENABLE = False` to recover the single-target baseline. The winner reports
auxiliary targets adding ~+0.001 CV.

For a quick notebook smoke test, set `FAST_DEV_RUN = True` (uses only a few recent
days per split); for comparable runs, keep it `False` and leave the split, features,
and metric unchanged.